# 📦 依存 wheel の実在確認

torch 2.13.0 / torchaudio 2.11.0 が cp313 linux で取れるか。


In [ ]:
import json
import subprocess
import sys
import urllib.request

PKGS = [("torch", "2.13.0"), ("torchaudio", "2.11.0"),
        ("torchvision", "0.28.0"), ("torchcodec", "0.14.0"),
        ("vllm", "0.32.0"), ("compressed-tensors", None)]


def files(pkg, ver):
    url = f"https://pypi.org/pypi/{pkg}/json" if ver is None else \
          f"https://pypi.org/pypi/{pkg}/{ver}/json"
    try:
        with urllib.request.urlopen(url, timeout=60) as r:
            d = json.load(r)
    except Exception as e:
        print(f"  {pkg} {ver}: 取得失敗 {type(e).__name__} {e}")
        return None
    if ver is None:
        ver = d["info"]["version"]
        print(f"  latest: {ver}")
    return d


print("=== PyPI wheel 実在チェック（linux x86_64 / cp313）===")
for pkg, ver in PKGS:
    print(f"\n-- {pkg} {ver}")
    d = files(pkg, ver)
    if not d:
        continue
    names = [u["filename"] for u in d["urls"]]
    print("   files:", len(names))
    hits = [n for n in names
            if ("cp313" in n or "py3-none" in n)
            and ("x86_64" in n or "manylinux" in n or "linux" in n or "any" in n)]
    for n in hits[:8]:
        print("    ", n)
    if not hits:
        print("     cp313/linux の wheel なし。全候補:", names[:6])

print("\n=== インストール済みバージョン（イメージ素の状態）===")
for mod in ("torch", "torchaudio", "torchvision"):
    r = subprocess.run([sys.executable, "-c",
                        f"import {mod}; print({mod}.__version__)"],
                       capture_output=True, text=True)
    print(f"  {mod}: {(r.stdout or r.stderr).strip()[:120]}")

print("\n=== pip の解決だけ確認（--dry-run --no-deps）===")
r = subprocess.run([sys.executable, "-m", "pip", "install", "--dry-run",
                    "--no-input", "--no-deps", "torchaudio==2.11.0"],
                   capture_output=True, text=True)
out = (r.stdout or "") + (r.stderr or "")
print("  rc:", r.returncode)
print("  ", " / ".join([l.strip() for l in out.splitlines() if l.strip()][-4:])[:400])
